## Get data from Zenodo

In [48]:
import subprocess
import os

output_dir = './data/'
os.makedirs(output_dir, exist_ok=True)

cmd1 = ["wget", "-c", "-P", output_dir, "https://zenodo.org/records/17143346/files/PE_O5Like_snr20.h5"] # PE file
cmd2 = ["wget", "-c", "-P", output_dir, "https://zenodo.org/records/17143346/files/injections_Ninj_2e7_O5Like_snr20.h5"] # inj file
subprocess.run(cmd1, capture_output=True, text=True)
subprocess.run(cmd2, capture_output=True, text=True)

CompletedProcess(args=['wget', '-c', '-P', './data/', 'https://zenodo.org/records/17143346/files/injections_Ninj_2e7_O5Like_snr20.h5'], returncode=0, stdout='', stderr='--2026-07-19 10:18:13--  https://zenodo.org/records/17143346/files/injections_Ninj_2e7_O5Like_snr20.h5\nResolving zenodo.org (zenodo.org)... 188.185.48.75, 188.184.103.118, 137.138.52.235, ...\nConnecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.\nHTTP request sent, awaiting response... 416 REQUESTED_RANGE_NOT_SATISFIABLE\n\n    The file is already fully retrieved; nothing to do.\n\n')

## Load data

In [49]:
import os, sys
import jax.numpy as jnp 

os.environ['CHIMERA_ENABLE_GPU'] = "False"
sys.path.append(os.getcwd()+'/../')
from CHIMERA import data as data

#file_ev   =  "./data/PE_O5Like_snr20.h5"
file_ev = '/home/mt/projects/gwtc-5/GWTC5_242CBC_FAR0.25_PE2048.h5'
theta_pe_det = data.load_gw_pe_samples(file_ev, 
                                 parameters=['m1det', 'm2det', 'dL'], 
                                 return_struct=True)
#pe_prior = theta_pe_det.dL**2
pe_prior = jnp.load('/home/mt/projects/gwtc-5/GWTC5_242CBC_FAR0.25_PE2048prior.npy')
theta_pe_det = theta_pe_det.update(pe_prior=pe_prior)
nev = len(theta_pe_det.dL)
print(f'Loaded {nev} GW events')

Loaded 242 GW events


## Instantiate population model

In [58]:
from CHIMERA.cosmo import flrw
from CHIMERA.mass.conditioned import plp
from CHIMERA.mass.paired import bpl_dip_two_peaks, bpl_dip_three_peaks
from CHIMERA.rate import madau_dickinson
from CHIMERA import population

# define population models with some fiducials
cosmo = flrw(H0 = 70., Om0=0.25, z_max = 5.)
mass = bpl_dip_two_peaks(m_low = 0.92, m_high = 125) #plp(m_low=0.5, m_high=150, lambda_peak=0.039, alpha=3.4, beta=1.1, delta_m=4.8, mu_g=34., sigma_g=3.6)
rate = madau_dickinson(gamma = 2.7, kappa =  3., zp = 2.)

population = population(cosmo, mass, rate, scale_free=True)

In [59]:
mass.as_dict

{'m_low': 0.92,
 'm_high': 125,
 'alpha_1': 2.1553739953196223,
 'alpha_2': 1.848352853466305,
 'beta_bottom': 1.500961280411781,
 'beta_top': 2.5908978874521305,
 'mu_g_low': 9.056012551383695,
 'sigma_g_low': 0.6948024105466937,
 'mu_g_high': 27.134217763853904,
 'sigma_g_high': 8.384405984258168,
 'lambda_g': 0.2651222683400193,
 'lambda_1': 0.7091553393032455,
 'bottomsmooth': 0.07554021010270083,
 'topsmooth': 0.0991675925989498,
 'leftdip': 2.2922222205817913,
 'rightdip': 6.875503912163882,
 'leftdipsmooth': 0.12602841524720307,
 'rightdipsmooth': 0.12531435,
 'deep': 0.4849206386665612}

## Instantiate the Selection Function module

In [60]:
from CHIMERA import selection_function

#file_inj = "./data/injections_Ninj_2e7_O5Like_snr20.h5"
file_inj = "/home/mt/projects/gwtc-5/GWTC5_injections_far0.25.h5"
theta_inj_det = data.load_injection_data(file_inj, snr_cut=None, key_mapping={'snr':'snr', 'log_pdraw':'log_pdraw'}, frame='detector')

sel_fcn = selection_function(theta_inj_det, N_inj = 1568035640.0) #N_inj=20*1e6)

## Instantiate the Hyperlike

In [61]:
from CHIMERA import hyperlikelihood

hyperlike = hyperlikelihood(  # data
  theta_gw_det = theta_pe_det,
  # population 
  population=population,
  # integration grid resolution
  z_grids_res = 300,  
  # selection function
  selection_function=sel_fcn,
  # numerical stability
  pe_neff = 2.0,
  inj_neff = None, # default to 5*Nev  
  # KDE settings
  kind_kde = 'fft',
  kernel = 'epan',
  kde_bw = None, # default to scott
  num_bins = 200,
)

2026-07-19 10:19:20,764 - CHIMERA - WARNING - `kde_bw` is None, using Scott rule for bandwidth.
2026-07-19 10:19:20,765 - CHIMERA - INFO - FFT KDE only supports Gaussian kernel. Setting to 'gaussian'
2026-07-19 10:19:20,766 - CHIMERA - INFO - Created hyperlikelihood model. Using 242 GW events.


## 1d on H0

In [ ]:
import numpy as np
import tqdm
import matplotlib.pyplot as plt

H0 = np.linspace(50, 90, 50)
log_like_evs_H0 = np.zeros((H0.shape[0], hyperlike.nevents))
loglike_H0 = np.zeros_like(H0)
N_exp_H0 = np.zeros_like(H0)

for i, h0 in tqdm.tqdm(enumerate(H0)):
  log_like_evs, N_exp, neff_inj, log_hyperlike = hyperlike.compute_all(H0=h0)
  log_like_evs_H0[i] = log_like_evs
  N_exp_H0[i] = N_exp
  loglike_H0[i] = log_hyperlike

# exponentiate and normalize
log_like_evs_H0 -= np.nanmax(log_like_evs_H0)
like_evs_H0 = np.exp(log_like_evs_H0) 
norms_evs = np.trapz(like_evs_H0, x=H0, axis=0)

loglike_H0 -= np.nanmax(loglike_H0)
like_H0 = np.exp(loglike_H0)
like_H0 /= np.trapz(like_H0, H0)

for e in range(hyperlike.nevents):
    lab = 'single event posteriors' if e == 0 else None
    plt.plot(H0, like_evs_H0[:,e]/norms_evs[e], c='gray', lw=0.75, alpha=0.75, label = lab )
plt.plot(H0, like_H0, label='posterior')
plt.axvline(70, ls='--', c='k', label= 'fiducial')
plt.legend()
plt.show()

29it [00:09,  2.36it/s]

In [55]:
import json

with open('/home/mt/downloads/icarogw_fullpop_spectral.json') as f:
    res = json.load(f)

for k in res['posterior']['content'].keys():
    print(k, np.median(res['posterior']['content'][k]))

H0 71.56745957566302
alpha_1 2.1553739953196223
alpha_2 1.848352853466305
beta_bottom 1.500961280411781
beta_top 2.5908978874521305
mu_g_low 9.056012551383695
sigma_g_low 0.6948024105466937
mu_g_high 27.134217763853904
sigma_g_high 8.384405984258168
lambda_g 0.2651222683400193
lambda_g_low 0.7091553393032455
mmin 0.961834913949035
mmax 96.86391602156012
bottomsmooth 0.07554021010270083
topsmooth 0.0991675925989498
leftdip 2.2922222205817913
rightdip 6.875503912163882
leftdipsmooth 0.12602841524720307
rightdipsmooth 0.1253143500776007
deep 0.4849206386665612
gamma 2.7119607706386923
kappa 2.9949269961522846
zp 2.4879652879458134
Om0 0.3065
log_likelihood -2552.3480441237125
log_prior -41.151081155092164


In [57]:
import json

with open('/home/mt/downloads/icarogw_bpl3p_spectral.json') as f:
    res = json.load(f)

for k in res['posterior']['content'].keys():
    print(k, np.median(res['posterior']['content'][k]))

H0 74.57953628634637
alpha_1 1.7723937807650858
alpha_2 5.063826317550281
beta_bottom 1.5835680997888992
beta_top 2.4359528976425393
mu_g_1 8.903700220355121
sigma_g_1 0.8391676748455468
mu_g_2 24.556312970528175
sigma_g_2 10.002284990279694
mu_g_3 62.73443845058334
sigma_g_3 8.683579188378397
lambda_g 0.3274896479007529
lambda_1 0.6348482960571541
lambda_2 0.9357685300341941
mmin 0.9245569825399548
mmax 124.20678928592034
bottomsmooth 0.08456654031744229
topsmooth 0.07731696292920931
leftdip 2.416688462245518
rightdip 8.017044339987327
leftdipsmooth 0.11437807134426872
rightdipsmooth 0.081615513156543
deep 0.5744362121512125
gamma 2.6649924665760474
kappa 2.97716364827715
zp 2.547626331272102
Om0 0.3065
log_likelihood -2551.118463093537
log_prior -47.502458785867645
